## Weekly Plan
### Week 3 (optional, if taking 3 weeks)
- Learn async with `asyncio` and `aiohttp`.
- Refactor scripts into modules and reusable helper functions.
- Add basic tests with `pytest`.

### Setup
Run this once to install required packages in the notebook environment.

In [1]:
%pip install aiohttp pytest -q

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inference-gpu 0.39.0 requires nvidia-ml-py<13.0.0, which is not installed.
inference-gpu 0.39.0 requires onnxruntime-gpu<1.20.0,>=1.15.1, which is not installed.
inference-gpu 0.39.0 requires opencv-contrib-python<=4.10.0.84,>=4.8.1.78, which is not installed.
inference-gpu 0.39.0 requires paho-mqtt~=1.6.1, which is not installed.
inference-gpu 0.39.0 requires py-cpuinfo~=9.0.0, which is not installed.
inference-gpu 0.39.0 requires pydot~=2.0.0, which is not installed.
inference-gpu 0.39.0 requires pylogix==1.0.5, which is not installed.
inference-gpu 0.39.0 requires pymodbus<=3.8.3,>=3.6.9, which is not installed.
inference-gpu 0.39.0 requires redis~=5.0.0, which is not installed.
inference-gpu 0.39.0 requires scikit-image<=0.24.0,>=0.19.0, which is not installed.
inference-gpu 0.39.0 requires shapely<2.1.0,>=2.0

### 1) Learn async with `asyncio` + `aiohttp`
Fetch multiple free endpoints concurrently from JSONPlaceholder.

In [4]:
import asyncio
import aiohttp
import time

BASE_URL = "https://jsonplaceholder.typicode.com"
POST_IDS = list(range(1, 17))

async def fetch_post(session: aiohttp.ClientSession, post_id: int) -> dict:
    url = f"{BASE_URL}/posts/{post_id}"
    async with session.get(url, timeout=20) as response:
        response.raise_for_status()
        return await response.json()

async def fetch_posts_concurrently(post_ids: list[int]) -> list[dict]:
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_post(session, pid) for pid in post_ids]
        return await asyncio.gather(*tasks)

start = time.perf_counter()
posts = await fetch_posts_concurrently(POST_IDS)
elapsed = time.perf_counter() - start

print(f"Fetched {len(posts)} posts in {elapsed:.2f}s")
print("Sample title:", posts[0]["title"])

Fetched 16 posts in 0.43s
Sample title: sunt aut facere repellat provident occaecati excepturi optio reprehenderit


### 2) Refactor into reusable helper functions
These functions are modular and can be reused in scripts or apps.

In [5]:
def normalize_post(post: dict) -> dict:
    return {
        "post_id": post.get("id"),
        "user_id": post.get("userId"),
        "title": str(post.get("title", "")).strip(),
        "body_preview": str(post.get("body", ""))[:60],
    }

def normalize_posts(posts: list[dict]) -> list[dict]:
    return [normalize_post(post) for post in posts]

def posts_to_user_summary(posts: list[dict]) -> dict[int, int]:
    summary: dict[int, int] = {}
    for post in posts:
        user_id = int(post.get("userId", 0))
        summary[user_id] = summary.get(user_id, 0) + 1
    return summary

normalized = normalize_posts(posts)
summary = posts_to_user_summary(posts)

print("Normalized sample:", normalized[0])
print("Posts per user (subset):", dict(list(summary.items())[:5]))

Normalized sample: {'post_id': 1, 'user_id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body_preview': 'quia et suscipit\nsuscipit recusandae consequuntur expedita e'}
Posts per user (subset): {1: 10, 2: 6}


### 3) Add basic tests with `pytest`
Write your helper functions and tests to files, then run `pytest`.

In [7]:
from pathlib import Path

module_code = '''
def normalize_post(post: dict) -> dict:
    return {
        "post_id": post.get("id"),
        "user_id": post.get("userId"),
        "title": str(post.get("title", "")).strip(),
        "body_preview": str(post.get("body", ""))[:60],
    }

def posts_to_user_summary(posts: list[dict]) -> dict[int, int]:
    summary: dict[int, int] = {}
    for post in posts:
        user_id = int(post.get("userId", 0))
        summary[user_id] = summary.get(user_id, 0) + 1
    return summary
'''

test_code = '''
from week3_helpers import normalize_post, posts_to_user_summary

def test_normalize_post_trims_title():
    data = {"id": 1, "userId": 2, "title": "  hello  ", "body": "abc"}
    normalized = normalize_post(data)
    assert normalized["title"] == "hello"
    assert normalized["post_id"] == 1

def test_posts_to_user_summary_counts_posts():
    posts = [
        {"userId": 1, "id": 1, "title": "a", "body": "x"},
        {"userId": 1, "id": 2, "title": "b", "body": "y"},
        {"userId": 2, "id": 3, "title": "c", "body": "z"},
    ]
    summary = posts_to_user_summary(posts)
    assert summary[1] == 2
    assert summary[2] == 1
'''

Path("week3_helpers.py").write_text(module_code, encoding="utf-8")
Path("test_week3_helpers.py").write_text(test_code, encoding="utf-8")
print("Created week3_helpers.py and test_week3_helpers.py")

Created week3_helpers.py and test_week3_helpers.py
